In [0]:
df_silver_customers=spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/customers")



In [0]:


# =========================================================================
# GOLD LAYER: dim_customers Worker Notebook
# =========================================================================

# 1. Configuration & Path Definitions (Hardcoded for self-contained execution)
GOLD_STORAGE_PATH = "abfss://gold@aze2estorage.dfs.core.windows.net/dimensions/dim_customers"
TABLE_NAME = "gold.dim_customers"

print(f"🔄 Starting build for {TABLE_NAME}...")
print(f"📂 Target Storage Path: {GOLD_STORAGE_PATH}")


print(f"📥 Read {df_silver_customers.count()} rows from silver.customers")

# 3. Select and Format Columns for Gold Consumer Readiness
# (Since silver.customers is already clean, we map/select the exact schema for BI)
gold_dim_customers = df_silver_customers.select(
    "customer_key",
    "prefix",
    "first_name",
    "last_name",
    "full_name",
    "birth_date",
    "age",
    "age_group",
    "gender",
    "marital_status",
    "email_address",
    "annual_income",
    "total_children",
    "education_level",
    "occupation",
    "is_home_owner"
)

# 4. Write Delta Files to the Explicit Gold Storage Path
gold_dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_STORAGE_PATH)

print("💾 Successfully wrote Delta files to storage path.")

# 5. Register/Update External Table in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_STORAGE_PATH}'
""")

print(f"🔗 Registered external table: {TABLE_NAME}")

# 6. Performance Optimization for Power BI DirectQuery / Joins
spark.sql(f"OPTIMIZE {TABLE_NAME} ZORDER BY (customer_key)")

print(f"⚡ Successfully optimized {TABLE_NAME} with Z-Ordering on customer_key.")
print(f"🎉 {TABLE_NAME} build complete!")
